In [1]:
from pathlib import Path
import csv

# Cell 1: Analysis only for Skin-Imperfections-9 (no file operations)
ROOT = Path('/media/aejaz/New Volume/Projects/DERMAVISION/dataset/Skin-Imperfections-9')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
SPLITS = ['train', 'valid', 'test']

print('Dataset:', ROOT)
print('-' * 70)

grand_total_images = 0

for split in SPLITS:
    split_dir = ROOT / split
    if not split_dir.exists():
        print(f"{split.upper()}: folder missing")
        continue

    image_files = [
        p for p in split_dir.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    n_images = len(image_files)
    grand_total_images += n_images

    print(f"{split.upper()} images: {n_images}")

    csv_path = split_dir / '_classes.csv'
    if csv_path.exists():
        with csv_path.open('r', encoding='utf-8', newline='') as f:
            rows = list(csv.reader(f))
        n_rows = len(rows)
        print(f"{split.upper()} _classes.csv rows (with header): {n_rows}")
        if n_rows > 0:
            print(f"{split.upper()} _classes.csv data rows: {n_rows - 1}")
    else:
        print(f"{split.upper()} _classes.csv: not found")

    print('-' * 70)

print('Grand total image files:', grand_total_images)

Dataset: /media/aejaz/New Volume/Projects/DERMAVISION/dataset/Skin-Imperfections-9
----------------------------------------------------------------------
TRAIN images: 6704
TRAIN _classes.csv rows (with header): 6705
TRAIN _classes.csv data rows: 6704
----------------------------------------------------------------------
VALID images: 1829
VALID _classes.csv rows (with header): 1830
VALID _classes.csv data rows: 1829
----------------------------------------------------------------------
TEST images: 751
TEST _classes.csv rows (with header): 752
TEST _classes.csv data rows: 751
----------------------------------------------------------------------
Grand total image files: 9284


In [1]:
from pathlib import Path
import random
import shutil

# Source folder: class-wise images (dark, light, mid-dark, mid-light)
SOURCE_ROOT = Path("/media/aejaz/New Volume/Projects/DERMAVISION/dataset/Skin-Tone-2")

# Output folder for split dataset
OUTPUT_ROOT = Path("/media/aejaz/New Volume/Projects/DERMAVISION/dataset/Skin-Tone-2-split")

# Split ratios (must sum to 1.0)
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

# Reproducibility seed
SEED = 42

# Image extensions to include
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Safety check
ratio_sum = TRAIN_RATIO + VALID_RATIO + TEST_RATIO
if abs(ratio_sum - 1.0) > 1e-9:
    raise ValueError(f"Split ratios must sum to 1.0, got {ratio_sum}")

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Source path not found: {SOURCE_ROOT}")

# Class names are taken from source subfolders
class_names = sorted([p.name for p in SOURCE_ROOT.iterdir() if p.is_dir()])
if not class_names:
    raise ValueError(f"No class folders found in {SOURCE_ROOT}")

print("Classes found:", class_names)

# Create destination tree train/valid/test/<class>
for split in ["train", "valid", "test"]:
    for cls in class_names:
        (OUTPUT_ROOT / split / cls).mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)
summary = {"train": {}, "valid": {}, "test": {}}

for cls in class_names:
    cls_dir = SOURCE_ROOT / cls
    images = [
        p for p in cls_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if not images:
        print(f"Skipping class '{cls}' (no matching images found).")
        summary["train"][cls] = 0
        summary["valid"][cls] = 0
        summary["test"][cls] = 0
        continue

    rng.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * TRAIN_RATIO)
    n_valid = int(n_total * VALID_RATIO)
    n_test = n_total - n_train - n_valid

    train_files = images[:n_train]
    valid_files = images[n_train:n_train + n_valid]
    test_files = images[n_train + n_valid:]

    for f in train_files:
        shutil.copy2(f, OUTPUT_ROOT / "train" / cls / f.name)
    for f in valid_files:
        shutil.copy2(f, OUTPUT_ROOT / "valid" / cls / f.name)
    for f in test_files:
        shutil.copy2(f, OUTPUT_ROOT / "test" / cls / f.name)

    summary["train"][cls] = len(train_files)
    summary["valid"][cls] = len(valid_files)
    summary["test"][cls] = len(test_files)

print("\nSplit completed at:", OUTPUT_ROOT)

print("\nImage counts per split and class")
for split in ["train", "valid", "test"]:
    split_total = 0
    print(f"\n{split.upper()}")
    for cls in class_names:
        count = summary[split].get(cls, 0)
        split_total += count
        print(f"  {cls}: {count}")
    print(f"  Total: {split_total}")

print("\nGrand total images:", sum(sum(summary[s].values()) for s in ["train", "valid", "test"]))

Classes found: ['dark', 'light', 'mid-dark', 'mid-light']

Split completed at: /media/aejaz/New Volume/Projects/DERMAVISION/dataset/Skin-Tone-2-split

Image counts per split and class

TRAIN
  dark: 6048
  light: 6838
  mid-dark: 7403
  mid-light: 4790
  Total: 25079

VALID
  dark: 1296
  light: 1465
  mid-dark: 1586
  mid-light: 1026
  Total: 5373

TEST
  dark: 1296
  light: 1466
  mid-dark: 1587
  mid-light: 1028
  Total: 5377

Grand total images: 35829


In [2]:
from pathlib import Path

# Cell 3: Analysis only for skin-type-1 (no file operations)
ROOT = Path('/media/aejaz/New Volume/Projects/DERMAVISION/dataset/skin-type-1')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
SPLITS = ['train', 'valid', 'test']

print('Dataset:', ROOT)
print('-' * 70)

grand_total_images = 0

for split in SPLITS:
    split_dir = ROOT / split
    if not split_dir.exists():
        print(f"{split.upper()}: folder missing")
        print('-' * 70)
        continue

    class_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()])
    print(f"{split.upper()} classes: {[c.name for c in class_dirs]}")

    split_total = 0
    for cls_dir in class_dirs:
        count = len([
            p for p in cls_dir.rglob('*')
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ])
        split_total += count
        print(f"  {cls_dir.name}: {count}")

    grand_total_images += split_total
    print(f"{split.upper()} total images: {split_total}")
    print('-' * 70)

print('Grand total image files:', grand_total_images)

Dataset: /media/aejaz/New Volume/Projects/DERMAVISION/dataset/skin-type-1
----------------------------------------------------------------------
TRAIN classes: ['combination', 'dry', 'normal']
  combination: 2013
  dry: 3219
  normal: 26
TRAIN total images: 5258
----------------------------------------------------------------------
VALID: folder missing
----------------------------------------------------------------------
TEST classes: ['combination', 'dry', 'normal', 'oily']
  combination: 99
  dry: 161
  normal: 145
  oily: 187
TEST total images: 592
----------------------------------------------------------------------
Grand total image files: 5850
